# Automated ML

TODO: Import Dependencies. In the cell below, import all the dependencies that you will need to complete the project.

In [1]:
from azureml.core import Workspace, Experiment
from azureml.train.automl import AutoMLConfig
from azureml.core.dataset import Dataset
from azureml.widgets import RunDetails

## Dataset

### Overview
TODO: In this markdown cell, give an overview of the dataset you are using. Also mention the task you will be performing.

The dataset contains data about the survival of patients with heart failure and some of there personal data like age and biomedical markers like serum creatinine.
The 

Full details about the dataset origin:

- Davide Chicco, Giuseppe Jurman: Machine learning can predict survival of patients with heart failure from serum creatinine and ejection fraction alone. BMC Medical Informatics and Decision Making 20, 16 (2020)
- Link to paper: https://doi.org/10.1186/s12911-020-1023-5
- Dataset obtained via Kaggle (https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data) under the CC BY 4.0 License (https://creativecommons.org/licenses/by/4.0/).

TODO: Get data. In the cell below, write code to access the data you will be using in this project. Remember that the dataset needs to be external.

In [2]:
ws = Workspace.from_config()

# choose a name for experiment
experiment_name = 'capstone-project'

experiment=Experiment(ws, experiment_name)

# Get the data (registered in the workspace by uploading the csv. details about origin, see the markdown cell above)
dataset_name = "heart-failure-clinical-data"
dataset = Dataset.get_by_name(ws, name=dataset_name)

In [3]:
# create a compute cluster
from azureml.core.compute import ComputeTarget, AmlCompute

cluster_name = "compute-cluster"
configuration = AmlCompute.provisioning_configuration(vm_size = "Standard_D4a_v4",
                                                      max_nodes=1,
                                                      vm_priority='dedicated',
                                                      idle_seconds_before_scaledown=600) # use for private azure account, west germany
cluster = ComputeTarget.create(workspace=ws,name=cluster_name,provisioning_configuration=configuration)

## AutoML Configuration

TODO: Explain why you chose the automl settings and cofiguration you used below.

In [17]:
# TODO: Put your automl settings here
automl_settings = {
    "experiment_timeout_hours": 1,
    "max_concurrent_iterations": 1, # limited due to quota
}

# TODO: Put your automl config here
automl_config = AutoMLConfig(
    task="classification",
    primary_metric="accuracy",
    training_data=dataset,
    label_column_name="DEATH_EVENT",
    n_cross_validations=4,
    enable_early_stopping=True,
    compute_target=cluster,
    **automl_settings)


In [18]:
# TODO: Submit your experiment
remote_run = experiment.submit(automl_config)

Submitting remote run.


Experiment,Id,Type,Status,Details Page,Docs Page
capstone-project,AutoML_c0e60528-988e-4770-ab8c-39e686cec87b,automl,NotStarted,Link to Azure Machine Learning studio,Link to Documentation


## Run Details

OPTIONAL: Write about the different models trained and their performance. Why do you think some models did better than others?

TODO: In the cell below, use the `RunDetails` widget to show the different experiments.

In [20]:
print(remote_run.get_portal_url())
#RunDetails(remote_run).show()

# since RunDetails doesn't seem to work with automl, use get_details instead.abs
remote_run.get_details()

https://ml.azure.com/runs/AutoML_c0e60528-988e-4770-ab8c-39e686cec87b?wsid=/subscriptions/d93ce4ff-fc1a-4ad9-a9c7-1df3382f954d/resourcegroups/resource-group/workspaces/azure-ml&tid=796de0a1-82b9-44ae-9691-75802bf973fa


{'runId': 'AutoML_c0e60528-988e-4770-ab8c-39e686cec87b',
 'target': 'compute-cluster',
 'status': 'Completed',
 'startTimeUtc': '2025-02-02T17:53:24.02285Z',
 'endTimeUtc': '2025-02-02T18:20:40.42051Z',
 'services': {},
 'warnings': [{'source': 'JasmineService',
   'message': 'No scores improved over last 10 iterations, so experiment stopped early. This early stopping behavior can be disabled by setting enable_early_stopping = False in AutoMLConfig for notebook/python SDK runs.'}],
 'properties': {'num_iterations': '1000',
  'training_type': 'TrainFull',
  'acquisition_function': 'EI',
  'primary_metric': 'accuracy',
  'train_split': '0',
  'acquisition_parameter': '0',
  'num_cross_validation': '4',
  'target': 'compute-cluster',
  'AMLSettingsJsonString': '{"path":null,"name":"capstone-project","subscription_id":"d93ce4ff-fc1a-4ad9-a9c7-1df3382f954d","resource_group":"resource-group","workspace_name":"azure-ml","region":"germanywestcentral","compute_target":"compute-cluster","spark_s

## Best Model

TODO: In the cell below, get the best model from the automl experiments and display all the properties of the model.



In [21]:
#TODO: Save the best model

remote_run.wait_for_completion()
best_run, fitted_model = remote_run.get_output() 

#import joblib
#joblib.dump(fitted_model, 'capstone-project/best_automl_model.pkl')
best_run.register_model(model_name='heart_failure_prediction_best_automl_model', model_path='outputs/model.pkl')

Model(workspace=Workspace.create(name='azure-ml', subscription_id='d93ce4ff-fc1a-4ad9-a9c7-1df3382f954d', resource_group='resource-group'), name=heart_failure_prediction_best_automl_model, id=heart_failure_prediction_best_automl_model:2, version=2, tags={}, properties={})

In [24]:
# display all the model properties
best_run_metrics = best_run.get_metrics()
parameter_values = best_run.get_details()['runDefinition']['arguments']

print('Best Run Id: ', best_run.id)
print('\n Accuracy:', best_run_metrics['accuracy'])

# print the model parameter values for the best run
print('\n Model parameters: ')
for elem in parameter_values:
    print(elem)

Best Run Id:  AutoML_c0e60528-988e-4770-ab8c-39e686cec87b_38

 Accuracy: 0.8695495495495495

 Model parameters: 


In [25]:
fitted_model

Pipeline(steps=[('datatransformer',
                 DataTransformer(enable_dnn=False, enable_feature_sweeping=True, is_cross_validation=True, working_dir='/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-notebook/code/Users/stephanherrmann')),
                ('prefittedsoftvotingclassifier',
                 PreFittedSoftVotingClassifier(classification_labels=array([0, 1]), estimators=[('27', Pipe...ype': 'cpu'}), reg_alpha=0, reg_lambda=0.20833333333333334, subsample=1, tree_method='auto'))]))], flatten_transform=False, weights=[0.08333333333333333, 0.08333333333333333, 0.08333333333333333, 0.08333333333333333, 0.08333333333333333, 0.08333333333333333, 0.16666666666666666, 0.08333333333333333, 0.08333333333333333, 0.08333333333333333, 0.08333333333333333]))])

## Model Deployment

Remember you have to deploy only one of the two models you trained but you still need to register both the models. Perform the steps in the rest of this notebook only if you wish to deploy this model.

TODO: In the cell below, register the model, create an inference config and deploy the model as a web service.

TODO: In the cell below, send a request to the web service you deployed to test it.

TODO: In the cell below, print the logs of the web service and delete the service

**Submission Checklist**
- I have registered the model.
- I have deployed the model with the best accuracy as a webservice.
- I have tested the webservice by sending a request to the model endpoint.
- I have deleted the webservice and shutdown all the computes that I have used.
- I have taken a screenshot showing the model endpoint as active.
- The project includes a file containing the environment details.
